In [1]:
#Agent Architecture
'''
User Input --> LLM Reasoning --> Tool Selection --> Tool Execution --> Structured Output --> Validation --> Retry (if required)
'''

'\nUser Input --> LLM Reasoning --> Tool Selection --> Tool Execution --> Structured Output --> Validation --> Retry (if required)\n'

In [2]:
#1.Import required libraries

In [3]:
import json
import math
from typing import List, Dict, Any

In [4]:
from pydantic import BaseModel, ValidationError, Field

In [5]:
#Pydantic is the most widely used data validation library for Python.

In [6]:
#We use Pydantic because production agent systems must validate data, not trust text.

In [7]:
#2.Define the Agent Output Schema (Structured Output)

In [8]:
class AgentOutput(BaseModel):
    answer: str
    confidence: float = Field(ge=0, le=1)
    sources: List[str]

In [9]:
#3.Define Tools

In [10]:
#3.1 Search Tool (Mock)
def search_tool(query: str) -> str:
    """
    Simulates a search tool.
    """
    return f"Search results for '{query}': AI Agents are systems that can reason and act using tools."

In [11]:
#3.2 Calculator Tool
def calculator_tool(expression: str) -> str:
    """
    Evaluates a mathematical expression.
    """
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Calculation error: {e}"

In [12]:
#3.3 External API Tool (Mock)
def weather_api_tool(city: str) -> Dict[str, Any]:
    """
    Simulates an external API call.
    """
    return {
        "city": city,
        "temperature": "22C",
        "condition": "Sunny"
    }

In [13]:
#4.Tool Registry (How the Agent Sees Tools)
#The LLM does not see Python functions.
#It sees descriptions.
TOOLS = {
    "search": {
        "description": "Use to fetch factual or up-to-date information.",
        "function": search_tool
    },
    "calculator": {
        "description": "Use to solve mathematical expressions.",
        "function": calculator_tool
    },
    "weather_api": {
        "description": "Use to get weather data for a city.",
        "function": weather_api_tool
    }
}

In [14]:
#5.Agent Prompt (Control Instructions)
SYSTEM_PROMPT = """
You are an AI Agent.
You may use tools if needed.
You must return a JSON object that strictly follows this schema:
{
  "answer": string,
  "confidence": number between 0 and 1,
  "sources": array of strings
}
If you call a tool, respond with:
TOOL_CALL: <tool_name>
INPUT: <json>

Otherwise, return the final JSON.
"""

In [15]:
#6.Simulated LLM Response #simulate LLM behavior.
def fake_llm(prompt: str, tool_used: bool) -> str:
    prompt_lower = prompt.lower()
    #Calculator decision
    if "calculate" in prompt_lower or "math" in prompt_lower:
        if tool_used:
            return json.dumps({
              "answer": "The result of the calculation 2 + 2 × 3 is 8.",
              "confidence": 1.0,
              "sources": ["calculator"]
        })
        else:
            return (
            "TOOL_CALL: calculator\n"
            "INPUT: {\"expression\": \"2 + 2 * 3\"}"
        )
    #Weather decision
    if "weather" in prompt_lower:
        if tool_used:
            return json.dumps({
            "answer": "The weather in Bangalore is sunny with a temperature of 22C.",
            "confidence": 0.85,
            "sources": ["weather_api"]
        })
        else:
            return (
            "TOOL_CALL: weather_api\n"
            "INPUT: {\"city\": \"Bangalore\"}"
        )
    #Search tool answer
    return json.dumps({
        "answer": "AI Agents are systems that reason and act using tools.",
        "confidence": 0.92,
        "sources": ["search_tool"]
    })

In [16]:
#7.Tool Execution Handler
def execute_tool(tool_name: str, tool_input: Dict[str, Any]) -> Any:
    tool = TOOLS[tool_name]["function"]
    return tool(**tool_input)

In [17]:
#8.Validation Logic
def validate_output(output: Dict[str, Any]) -> bool:
    try:
        AgentOutput(**output)
        return True
    except ValidationError as e:
        print("Validation error:", e)
        return False

In [18]:
#9.The Agent Loop
def run_agent(user_query: str, max_retries: int = 3) -> AgentOutput:
    prompt = SYSTEM_PROMPT + f"\nUser: {user_query}\n" 
    tool_used = False
    for attempt in range(max_retries):
        print(f"\nAttempt {attempt + 1}")        
        response = fake_llm(prompt, tool_used)         
        #Tool call detected
        if response.startswith("TOOL_CALL"):
            lines = response.splitlines()
            tool_name = lines[0].split(":")[1].strip()
            tool_input = json.loads(lines[1].split("INPUT:")[1])            
            tool_result = execute_tool(tool_name, tool_input)
            prompt += f"\nTool result: {tool_result}\n"
            tool_used = True
            continue        
        #Final output
        try:
            parsed = json.loads(response)            
        except json.JSONDecodeError:
            prompt += "\nYour output was not valid JSON. Fix it.\n"
            continue        
        if validate_output(parsed):
            return AgentOutput(**parsed)
        else:
            prompt += "\nYour output violated the schema. Fix it.\n"    
    raise RuntimeError("Agent failed after maximum retries")

In [19]:
#10. Run the Agent
result = run_agent("Explain AI Agents")
print(result)


Attempt 1
answer='AI Agents are systems that reason and act using tools.' confidence=0.92 sources=['search_tool']


In [20]:
#weather example
result = run_agent("What is the weather in Bangalore?")
print(result)


Attempt 1

Attempt 2
answer='The weather in Bangalore is sunny with a temperature of 22C.' confidence=0.85 sources=['weather_api']


In [21]:
#weather example
result = run_agent("Calculate 2 + 2 * 3")
print(result)


Attempt 1

Attempt 2
answer='The result of the calculation 2 + 2 × 3 is 8.' confidence=1.0 sources=['calculator']
